# Final models

The two outputs, refitted once on every development season and saved for use
outside the notebooks (a dashboard, re-scoring new pitches):

| output | in-play valuation | hitter features | trained on |
|---|---|---|---|
| **generic** | event classifier | none | 2021–2025 |
| **personalized** | event classifier, swing value shrunk toward generic (k = 0.6846) | hot zone, whiff/foul priors | 2022–2025 (2021 is prior data only) |

They score **2026**, the first ABS season. 2026 is a *previously inspected*
season — earlier versions of the project looked at it — so its results here are
an out-of-regime evaluation, not a clean test; the confirmatory test is 2027.
Nothing in the design is changed because of what 2026 shows. When 2027 is
complete, the same cells refit through 2026.

Models are saved to `models/` (gitignored) with `decomposition.save_models`;
the personalized one also records its shrink (`save_shrink`), so
`load_output(models_dir, name)` returns it already shrunk toward the generic
model. The 2026 scores below come from the reloaded files, so the saved models
are what they describe.

In [1]:
import sys; sys.path.insert(0, '..')
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import predict_both
from src.decomposition import (fit_decomposed, save_models, save_shrink, load_output,
                               PERSONALIZED_SHRINK)

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
SCORE_SEASON = 2026
TRAIN = {'generic': [2021, 2022, 2023, 2024, 2025], 'personalized': [2022, 2023, 2024, 2025]}
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']
PRIORS = ['prior_whiff', 'prior_foul']
FEATURES = {'generic': V3_FEATURES, 'personalized': V3_FEATURES + ['hot_zone'] + PRIORS}
MODEL_DIR = Path('..') / 'models'

In [2]:
df = D.drop_pitchers_batting(D.load_seasons(range(2021, SCORE_SEASON + 1), verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')

# Hitter priors, each season from the two before it (2026's from 2024-25).
key = ['game_pk', 'at_bat_number', 'pitch_number']
pdf = F.season_hot_zone(df).merge(F.season_contact_priors(df)[key + PRIORS],
                                  on=key, how='left', validate='one_to_one')
frames = {'generic': df, 'personalized': pdf}
print({name: f'{len(f):,} pitches' for name, f in frames.items()})

{'generic': '4,228,826 pitches', 'personalized': '3,539,008 pitches'}


## Fit and save

In [3]:
for name in FEATURES:
    feats, seasons = FEATURES[name], TRAIN[name]
    frame = frames[name].dropna(subset=[f for f in feats if f != 'count'])
    rv = D.run_value_table(frame, seasons)                     # run values from the training seasons only
    train = D.apply_run_value(frame[frame.season.isin(seasons)], rv, name='target').dropna(subset=['target'])
    model = fit_decomposed(train, feats, rv, in_play='classifier')
    save_models(model, MODEL_DIR / name, meta={
        'output': name, 'trained_on': seasons, 'features': feats, 'fitted': date.today().isoformat(),
        'score': DEC.DEFAULT_SCORE})
    rv.rename('run_value').to_frame().to_parquet(MODEL_DIR / name / 'run_value_table.parquet')
    print(f'{name:13s} trained on {seasons[0]}-{seasons[-1]}: {len(train):,} pitches, rounds {model.rounds}')

# The personalized output keeps a share k of what personalization adds (v6_shrink.ipynb).
save_shrink(MODEL_DIR / 'personalized', PERSONALIZED_SHRINK)
print(f'personalized: shrunk toward generic, k = {PERSONALIZED_SHRINK}')
sorted(str(p.relative_to(MODEL_DIR)) for p in MODEL_DIR.rglob('*') if p.is_file())

generic       trained on 2021-2025: 3,517,660 pitches, rounds {'take': 333, 'outcome': 787, 'bip_value': 162}


personalized  trained on 2022-2025: 2,827,844 pitches, rounds {'take': 357, 'outcome': 734, 'bip_value': 223}
personalized: shrunk toward generic, k = 0.6846


['generic/bip_value.ubj',
 'generic/meta.json',
 'generic/outcome.ubj',
 'generic/re_in_play.parquet',
 'generic/re_swing.parquet',
 'generic/run_value_table.parquet',
 'generic/take.ubj',
 'hitter_scores_2026.parquet',
 'personalized/bip_value.ubj',
 'personalized/meta.json',
 'personalized/outcome.ubj',
 'personalized/re_in_play.parquet',
 'personalized/re_swing.parquet',
 'personalized/run_value_table.parquet',
 'personalized/shrink.json',
 'personalized/take.ubj',
 'pitch_values_2026.parquet']

## Score 2026 with the saved models

Reloaded from disk. The 2026 target uses each model's training-season run-value
table, as every held-out season did.

In [4]:
scored = {}
for name in FEATURES:
    feats = FEATURES[name]
    model = load_output(MODEL_DIR, name)            # personalized comes back shrunk
    rv = pd.read_parquet(MODEL_DIR / name / 'run_value_table.parquet')['run_value']
    frame = frames[name].dropna(subset=[f for f in feats if f != 'count'])
    train = D.apply_run_value(frame[frame.season.isin(TRAIN[name])], rv, name='target')
    count_only = train.groupby(['swing', 'count'], observed=True)['target'].mean()
    s26 = D.apply_run_value(frame[frame.season == SCORE_SEASON], rv, name='target').dropna(subset=['target'])
    s26 = DEC.add_scores(predict_both(s26, model)).join(model.components(s26))
    s26['count_only'] = count_only.reindex(pd.MultiIndex.from_arrays([s26['swing'], s26['count']])).to_numpy()
    scored[name] = E.FoldRun(held=s26, models={SCORE_SEASON: model}, folds=())
{name: f'{len(r.held):,} pitches' for name, r in scored.items()}

{'generic': '711,016 pitches', 'personalized': '711,016 pitches'}

## How the models do on 2026 (out-of-regime evaluation)

Accuracy against the count-only predictor, and calibration grouped by predicted
value, as for every held-out season.

In [5]:
pd.concat({name: E.accuracy(r) for name, r in scored.items()}, names=['output'])[
    ['pitches', 'improvement_%', 'diff_lo', 'diff_hi']].round(5)

pitches  improvement_%  diff_lo  diff_hi
output       action season                                          
generic      take   2026     371986       74.30762 -0.00446 -0.00432
                    pooled   371986       74.30762 -0.00446 -0.00432
             swing  2026     339030        1.97748 -0.00183 -0.00169
                    pooled   339030        1.97748 -0.00183 -0.00169
personalized take   2026     371986       74.51283 -0.00442 -0.00428
                    pooled   371986       74.51283 -0.00442 -0.00428
             swing  2026     339030        2.17727 -0.00202 -0.00185
                    pooled   339030        2.17727 -0.00202 -0.00185

In [6]:
pd.concat({name: E.calibration_table(r) for name, r in scored.items()}, names=['output']).round(3)

slope  slope_lo  slope_hi
output       action season                           
generic      take   2026    1.016     1.012     1.019
             swing  2026    1.003     0.979     1.023
personalized take   2026    1.016     1.013     1.020
             swing  2026    0.996     0.973     1.017

In [7]:
# The score's behaviour in 2026: construct partials, split-half, Zone%.
rows = []
for name, r in scored.items():
    sc = E.player_metric(r.held, DEC.DEFAULT_SCORE)
    cv = E.construct_validity(r.held, sc).iloc[0]
    rows.append({'output': name, 'hitters': len(sc),
                 'chase | zone_swing': cv['chase | zone_swing'], 'zone_swing | chase': cv['zone_swing | chase'],
                 'split-half r': E.split_half(r.held, DEC.DEFAULT_SCORE).r_spearman_brown.mean(),
                 'Zone% r': E.zone_pct_correlation(r.held, sc).r.mean()})
pd.DataFrame(rows).set_index('output').round(3)

,hitters,chase | zone_swing,zone_swing | chase,split-half r,Zone% r
output,,,,,
generic,434,-0.893,0.659,0.800,0.267
personalized,434,-0.855,0.558,0.815,0.206


## Save the scores

Per-pitch values and hitter-season scores, as parquet, for the dashboard.

In [8]:
pitch_cols = ['game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'batter', 'pitcher', 'stand',
              'p_throws', 'count', 'pitch_type', 'plate_x_mid', 'plate_z_mid', 'swing', 'outcome']
value_cols = ['q_take', 'q_swing', 'edge', 'signed_edge']
pitches = scored['generic'].held[[c for c in pitch_cols if c in scored['generic'].held] + value_cols]
pitches = pitches.merge(scored['personalized'].held[['game_pk', 'at_bat_number', 'pitch_number'] + value_cols],
                        on=['game_pk', 'at_bat_number', 'pitch_number'], how='left', suffixes=('_generic', '_personalized'))

hitters = (E.player_metric(scored['generic'].held, DEC.DEFAULT_SCORE)
             .rename(columns={'raw': 'raw_generic', 'decision_value': 'generic'})
             .merge(E.player_metric(scored['personalized'].held, DEC.DEFAULT_SCORE)
                      .rename(columns={'raw': 'raw_personalized', 'decision_value': 'personalized'})
                      .drop(columns='pitches'), on=['season', 'batter'], how='left'))
try:
    from pybaseball import playerid_reverse_lookup
    nm = playerid_reverse_lookup(hitters.batter.tolist())
    nm['name'] = (nm.name_first + ' ' + nm.name_last).str.title()
    hitters = hitters.merge(nm[['key_mlbam', 'name']], left_on='batter', right_on='key_mlbam', how='left').drop(columns='key_mlbam')
except Exception as exc:
    print(f'name lookup unavailable ({exc})')

pitches.to_parquet(MODEL_DIR / f'pitch_values_{SCORE_SEASON}.parquet', index=False)
hitters.to_parquet(MODEL_DIR / f'hitter_scores_{SCORE_SEASON}.parquet', index=False)
print(f'{len(pitches):,} pitches, {len(hitters)} qualified hitters saved')
print(f"generic vs personalized: r = {np.corrcoef(hitters.generic, hitters.personalized.fillna(hitters.generic))[0, 1]:.3f}")
cols = [c for c in ['name', 'pitches', 'generic', 'personalized'] if c in hitters]
display(hitters.nlargest(10, 'personalized')[cols].round(1))
display(hitters.nsmallest(10, 'personalized')[cols].round(1))

Gathering player lookup table. This may take a moment.


711,016 pitches, 434 qualified hitters saved
generic vs personalized: r = 0.963


,name,pitches,generic,personalized
89,Gleyber Torres,1774,124.1,124.0
46,Corey Seager,1605,114.2,122.1
144,Juan Soto,1872,116.6,121.7
189,Will Smith,1179,119.1,120.8
398,Nick Kurtz,1845,119.4,120.6
170,Jj Bleday,2145,117.7,120.1
121,Kyle Tucker,2458,117.3,119.9
255,Spencer Torkelson,2592,118.9,119.8
248,Miguel Vargas,2781,121.2,119.6
23,Aaron Judge,1217,112.8,119.6


,name,pitches,generic,personalized
30,Javier Báez,691,59.8,62.3
64,Edmundo Sosa,898,68.5,67.3
282,Angel Martínez,1491,73.8,73.1
280,Noelvi Marte,761,74.7,74.7
58,Tyler Heineman,693,79.8,75.1
33,Trevor Story,982,74.1,75.5
133,Kyle Isbel,1147,75.7,76.6
396,Nathan Church,1788,80.3,77.5
384,Héctor Rodríguez,560,80.1,77.6
231,Ernie Clement,1915,75.2,78.2


## Findings — 2026, out of regime

**The models carry over to the ABS season.** On 2026 the swing models beat a
count-only predictor by 1.98% (generic) and 2.18% (personalized, shrunk) of MSE
— in line with the held-out 2023–25 seasons — and the take models by 74%, more
than the ~70% of earlier seasons, as expected when calls follow a fixed zone.

**Both are calibrated.** Swing slopes are 1.003 (generic, interval
0.979–1.023) and 0.996 (personalized, 0.973–1.017): the shrink toward the
generic model removes the over-spread the unshrunk personalized model showed
(0.948).

**The score in 2026.** Split-half reliability is 0.800 (generic) and 0.815
(personalized). Construct partials are lower than in the held-out seasons —
−0.893 / +0.659 generic, −0.855 / +0.558 personalized — and Zone% correlates
0.267 and 0.206. The two outputs' hitter scores correlate 0.963.

These are reported, not acted on: 2026 was inspected by earlier versions, so it
is an out-of-regime evaluation, and nothing in the design changes because of it.
The saved models and 2026 scores are in `models/`; `load_output(models_dir,
'personalized')` returns the personalized model already shrunk.